# BigSMILES and BigSmirk

<a href="https://colab.research.google.com/github/BattModels/smirk/blob/main/docs/smirk_demo.ipynb">
    <img alt="Open In Colab" src="https://colab.research.google.com/assets/colab-badge.svg">
</a>
<a href="https://mybinder.org/v2/gh/BattModels/smirk/main?urlpath=%2Fdoc%2Ftree%2Fdocs%2Fsmirk_demo.ipynb">
    <img alt="Binder" src="https://mybinder.org/badge_logo.svg">
</a>



BigSmirk tokenizes the [BigSMILES] encoding for macromolecules all the way down to their constituent elements.

Let's see it in action!

[BigSMILES]: https://olsenlabmit.github.io/BigSMILES/docs/#the-bigsmiles-project

🐍 Installation is easy with pre-build binaries on [PyPI](https://pypi.org/project/smirk/) and [GitHub](https://github.com/BattModels/smirk/releases). Just run: `pip install smirk`

> Installing from source? See [installing from source](./developer.md#installing-from-source) for instructions.

In [ ]:
!python -m pip install smirk transformers

## First steps

🤗 smirk subclasses Hugging Face's [PreTrainedTokenizerBase](#transformers.PreTrainedTokenizerBase) for seamless compatibility and leverages [Tokenizers] for raw rust-powered speed. No need to learn another framework; everything works out of the box 🎁

[Tokenizers]: https://huggingface.co/docs/tokenizers/index

In [2]:
from smirk import SmirkBigSmilesFast

# Just import, set the bigsmiles argument and tokenize!
bigsmirk = SmirkBigSmilesFast()
bigsmirk("{[][$]CC[$],[$]CC(CC)[$][]}") # ethylene butene copolymer

{'input_ids': [159, 148, 150, 148, 2, 150, 45, 45, 148, 2, 150, 161, 148, 2, 150, 45, 45, 4, 45, 45, 5, 148, 2, 150, 148, 150, 160], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
# Batch Tokenization with Padding
batch = bigsmirk([
    "[H]O{[>][<]C(=O)CCCCC(=O)[<],[>]NCCCCCCN[>][<]}[H]", # nylon-6,6
    "{[][<]OCC[>][<]}{[>][<]OC(C)C[>][]}", # block copolymer
    "{[][<]C(=O)c1ccc(cc1)C(=O)[<],[>]OCCO[>][]}", # alternation co-polymer
], padding="longest")
batch

{'input_ids': [[148, 71, 150, 102, 159, 148, 164, 150, 148, 163, 150, 45, 4, 22, 102, 5, 45, 45, 45, 45, 45, 4, 22, 102, 5, 148, 163, 150, 161, 148, 164, 150, 93, 45, 45, 45, 45, 45, 45, 93, 148, 164, 150, 148, 163, 150, 160, 148, 71, 150], [159, 148, 150, 148, 163, 150, 102, 45, 45, 148, 164, 150, 148, 163, 150, 160, 159, 148, 164, 150, 148, 163, 150, 102, 45, 4, 45, 5, 45, 148, 164, 150, 148, 150, 160, 168, 168, 168, 168, 168, 168, 168, 168, 168, 168, 168, 168, 168, 168, 168], [159, 148, 150, 148, 163, 150, 45, 4, 22, 102, 5, 153, 12, 153, 153, 153, 4, 153, 153, 12, 5, 45, 4, 22, 102, 5, 148, 163, 150, 161, 148, 164, 150, 102, 45, 45, 102, 148, 164, 150, 148, 150, 160, 168, 168, 168, 168, 168, 168, 168], [159, 163, 10, 45, 22, 45, 10, 153, 12, 153, 153, 4, 45, 45, 45, 45, 45, 45, 5, 153, 4, 153, 153, 12, 45, 45, 45, 45, 45, 45, 5, 10, 45, 22, 45, 10, 153, 13, 153, 153, 153, 4, 153, 153, 13, 5, 164, 160, 168, 168]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [19]:
# Back to polymers!
bigsmirk.batch_decode(batch["input_ids"], skip_special_tokens=True)

['[H]O{[>][<]C(=O)CCCCC(=O)[<],[>]NCCCCCCN[>][<]}[H]',
 '{[][<]OCC[>][<]}{[>][<]OC(C)C[>][]}',
 '{[][<]C(=O)c1ccc(cc1)C(=O)[<],[>]OCCO[>][]}',
 '{</C=C/c1cc(CCCCCC)c(cc1CCCCCC)/C=C/c2ccc(cc2)>}']

## Zero to Polymer Foundation Model with Smirk!

Let's train a small [RoBERTa] model on polymers from [S. Choi et al., 2024] using Hugging Face and smirk.

[RoBERTa]: https://doi.org/10.48550/ARXIV.1907.11692
[S. Choi et al., 2024]:https://www.nature.com/articles/s41597-024-03212-4

In [ ]:
!python -m pip install accelerate datasets torch

### Dataset Preprocessing

In [15]:
print(r.status_code, r.headers.get("Content-Type"), r.url, r.text[:500])

202 text/html; charset=UTF-8 https://springernature.figshare.com/ndownloader/files/42507037 


In [14]:
from datasets import load_dataset

import tempfile
import zipfile
import requests
import os
d=tempfile.mkdtemp()
z=os.path.join(d,"with_Tg.zip")
url = "https://springernature.figshare.com/ndownloader/files/42507037"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers, allow_redirects=True, timeout=60)
r.raise_for_status()
with open(z, "wb") as f:
    f.write(r.content)

with zipfile.ZipFile(z) as zf:
    zf.extractall(d)
dataset=load_dataset("csv", data_files=[os.path.join(d,"JCIM_sup_bigsmiles.csv"), os.path.join(d,"Bicerano_bigsmiles.csv")])["train"].select_columns(["BigSMILES"]).train_test_split(test_size=0.2)
dataset=dataset.map(bigsmirk, input_columns=["BigSMILES"], desc="Tokenizing")

BadZipFile: File is not a zip file

> 💡 huggingface/tokenizers may raise a warning about being forked as we've already used our tokenizers (this isn't a smirk issue).
> It's harmless, but when actually training it's best to avoid tokenization until after the fork to benefit from the rust-level parallelism

🎉 That's it! We've tokenized all of QM9 using smirk!

In [ ]:
dataset["train"].to_pandas().head()

### Training
Once we've tokenized the dataset, training the model is just a matter of configuration.

In [ ]:
from accelerate import Accelerator
from transformers import Trainer, TrainingArguments, RobertaForMaskedLM, RobertaConfig, DataCollatorForLanguageModeling

# A very small model for demonstrating training a molecular foundation model with smirk 
config = RobertaConfig(
    vocab_size=len(smirk),
    hidden_size=256,
    intermediate_size=1024,
    num_hidden_layers=4,
    num_attention_heads=4,
)
model = RobertaForMaskedLM(config)

# Setup up the trainer to use our dataset
trainer = Trainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=smirk,
    data_collator=DataCollatorForLanguageModeling(smirk), # The data collator needs to know about our tokenizer
)

In [ ]:
trainer.train()